# 05_model_stacking – Advanced Ensembles with Stacking

**Goal:** Combine multiple base models (Ridge, LightGBM, XGBoost, Random Forest) using a meta‑learner to improve prediction accuracy. Stacking (or stacked generalization) uses out‑of‑fold predictions from base models as features for a second‑level model.

**References:**
- [Stacking (machine learning) – Wikipedia](https://en.wikipedia.org/wiki/Ensemble_learning#Stacking)
- Wolpert, D. H. (1992). "Stacked generalization." *Neural Networks*.

---

## 1. Setup and Imports

We'll import necessary libraries, including XGBoost (install with `pip install xgboost` if needed). We'll also reuse our custom modules for data loading and feature preprocessing (the same `FeatureTransformer` from `04_modeling_baseline.ipynb`).

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder

from src import config
from src.data import load_train, load_test
from src.features import FeatureTransformer, log_transform_target, inverse_log_transform
from src.utils import rmse

# For reproducibility
RANDOM_STATE = 42
N_SPLITS = 10

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

We load the raw training data, remove the known outliers, and separate features and target. We'll work with the log-transformed target throughout this notebook.

In [2]:
# Load data
df = load_train()
print(f"Original train shape: {df.shape}")

# Remove known outliers (as before)
outlier_idx = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)].index
df = df.drop(outlier_idx)
print(f"Shape after outlier removal: {df.shape}")

# Target and features
y_raw = df[config.TARGET]
X_raw = df.drop([config.TARGET, "Id"], axis=1)

# Log transform target
y = log_transform_target(y_raw)

Original train shape: (1460, 81)
Shape after outlier removal: (1458, 81)


## 3. Define Base Models

We'll use four diverse base models:

- **Ridge Regression** (linear, with L2 penalty)  
- **Random Forest** (bagging, handles non-linearities)  
- **LightGBM** (gradient boosting, fast)  
- **XGBoost** (another boosting algorithm)  

We'll define them with sensible default parameters (you may tune later).

In [3]:
base_models = {
    'ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'rf': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'lgb': lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.01, random_state=RANDOM_STATE, n_jobs=-1),
    'xgb': xgb.XGBRegressor(n_estimators=5000, learning_rate=0.01, random_state=RANDOM_STATE, n_jobs=-1)
}

## 4. Generate Out-of-Fold Predictions for Each Base Model

We'll perform 10-fold cross-validation. For each fold, we:

- Fit the `FeatureTransformer` on the training fold.  
- Transform both training and validation folds.  
- Train each base model on the transformed training data.  
- Predict on the validation fold and store the predictions.  

We'll collect OOF predictions for each model in a dictionary:  
`oof_preds[model_name]` will be an array of length `len(X_raw)`.

In [4]:
# 4. Generate Out-of-Fold Predictions for Each Base Model

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Dictionary to store OOF predictions for each model
oof_preds = {name: np.zeros(len(X_raw)) for name in base_models.keys()}
fold_models = []   # list of dicts, one per fold

for fold, (train_idx, val_idx) in enumerate(kf.split(X_raw)):
    print(f"\n=== Fold {fold+1}/{N_SPLITS} ===")
    X_train, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Feature transformation
    ft = FeatureTransformer()
    ft.fit(X_train)
    X_train_t = ft.transform(X_train)
    X_val_t = ft.transform(X_val)

    # One‑hot encode categorical columns
    num_cols = X_train_t.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train_t.select_dtypes(include=['category', 'object']).columns.tolist()

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_cat = ohe.fit_transform(X_train_t[cat_cols])
    X_val_cat = ohe.transform(X_val_t[cat_cols])

    X_train_num = X_train_t[num_cols].values
    X_val_num = X_val_t[num_cols].values
    X_train_proc = np.hstack([X_train_num, X_train_cat])
    X_val_proc = np.hstack([X_val_num, X_val_cat])

    fold_dict = {
        'ft': ft,
        'ohe': ohe,
        'num_cols': num_cols,
        'cat_cols': cat_cols
    }

    # Train each base model
    for name, model in base_models.items():
        # Create a fresh model instance
        if name == 'ridge':
            m = Ridge(alpha=1.0, random_state=RANDOM_STATE)
        elif name == 'rf':
            m = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
        elif name == 'lgb':
            m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.01, random_state=RANDOM_STATE, n_jobs=-1)
        elif name == 'xgb':
            m = xgb.XGBRegressor(n_estimators=5000, learning_rate=0.01, random_state=RANDOM_STATE, n_jobs=-1)

        # Fit with early stopping for boosting models
        if name == 'lgb':
            m.fit(X_train_proc, y_train,
                  eval_set=[(X_val_proc, y_val)],
                  early_stopping_rounds=200,
                  verbose=False)
        elif name == 'xgb':
            # Try with early_stopping_rounds; if fails (older XGBoost), fit without it
            try:
                m.fit(X_train_proc, y_train,
                      eval_set=[(X_val_proc, y_val)],
                      early_stopping_rounds=200,
                      verbose=False)
            except TypeError:
                m.fit(X_train_proc, y_train,
                      eval_set=[(X_val_proc, y_val)],
                      verbose=False)
        else:
            m.fit(X_train_proc, y_train)

        preds = m.predict(X_val_proc)
        oof_preds[name][val_idx] = preds
        fold_dict[name] = m

        rmse_fold = np.sqrt(mean_squared_error(y_val, preds))
        print(f"  {name} fold RMSE: {rmse_fold:.5f}")

    fold_models.append(fold_dict)

# Overall OOF performance
print("\n=== Base Model OOF Performance ===")
for name in base_models.keys():
    overall_rmse = np.sqrt(mean_squared_error(y, oof_preds[name]))
    print(f"{name}: OOF log RMSE = {overall_rmse:.5f}")

INFO:src.features:Skewed features to transform: ['MSSubClass', 'LotArea', '1stFlrSF', 'GrLivArea']



=== Fold 1/10 ===
  ridge fold RMSE: 0.08943
  rf fold RMSE: 0.13906
  lgb fold RMSE: 0.12905


INFO:src.features:Skewed features to transform: ['MSSubClass', 'LotArea', '1stFlrSF', 'GrLivArea', 'KitchenAbvGr']


  xgb fold RMSE: 0.13067

=== Fold 2/10 ===
  ridge fold RMSE: 0.13450
  rf fold RMSE: 0.14436
  lgb fold RMSE: 0.13662


INFO:src.features:Skewed features to transform: ['MSSubClass', 'LotArea', '1stFlrSF', 'GrLivArea']


...

  xgb fold RMSE: 0.12865

=== Fold 8/10 ===
  ridge fold RMSE: 0.10661
  rf fold RMSE: 0.12432
  lgb fold RMSE: 0.12306


INFO:src.features:Skewed features to transform: ['MSSubClass', 'LotArea', '1stFlrSF', 'GrLivArea']


  xgb fold RMSE: 0.12155

=== Fold 9/10 ===
  ridge fold RMSE: 0.10820
  rf fold RMSE: 0.12993
  lgb fold RMSE: 0.11950


INFO:src.features:Skewed features to transform: ['MSSubClass', 'LotArea', '1stFlrSF', 'GrLivArea']


  xgb fold RMSE: 0.11917

=== Fold 10/10 ===
  ridge fold RMSE: 0.08963
  rf fold RMSE: 0.11299
  lgb fold RMSE: 0.10465
  xgb fold RMSE: 0.10503

=== Base Model OOF Performance ===
ridge: OOF log RMSE = 0.11341
rf: OOF log RMSE = 0.13282
lgb: OOF log RMSE = 0.12402
xgb: OOF log RMSE = 0.12378


## 5. Train a Meta-Model on OOF Predictions

The OOF predictions become the training features for the meta-model. We'll use a simple **Ridge regression** as the meta-model (you could also use a small LightGBM model).

###  Important: Avoiding Optimistic Evaluation

After generating OOF predictions from the base models:

1. For each fold *i*, base models were trained on all data except fold *i* and predicted on fold *i*.
2. After all folds, we have OOF predictions for the entire dataset.
3. We can now train a meta-model using:
   - Features → OOF predictions
   - Target → True labels

However, evaluating the meta-model on the same OOF matrix is slightly optimistic because the meta-model sees features generated from models trained on overlapping data.

To get a more reliable estimate, we need a **second layer of cross-validation**.

---

## Proper Stacking Procedure (Second-Level CV)

We will:

- Use the same fold indices (or define new folds).
- For each fold *i*:
  - Train the meta-model on the OOF predictions from all folds except *i*.
  - Predict on fold *i*.
- Collect second-level OOF predictions.
- Compute RMSE on these predictions.

This gives a properly cross-validated stacking score.

---

## Implementation Plan

We'll define a function that:

- Takes:
  - `oof_preds` (dictionary of base model OOF predictions)
  - `y` (true targets)
  - `folds` (cross-validation split indices)
- Constructs the meta-feature matrix
- Performs second-level cross-validation
- Returns:
  - Meta-model OOF predictions
  - Cross-validated RMSE

This ensures our stacking pipeline remains statistically sound and avoids data leakage.

In [5]:
# We already have the fold indices from kf.split (the same as before)
# We'll now perform a second-level CV: for each fold, train the meta-model on OOF predictions from the other folds,
# and predict on the current fold's OOF predictions.

meta_train_features = np.column_stack([oof_preds[name] for name in base_models.keys()])  # shape (n_samples, n_models)

# We'll store meta predictions
meta_oof = np.zeros(len(y))
meta_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_raw)):
    X_meta_train = meta_train_features[train_idx]
    X_meta_val = meta_train_features[val_idx]
    y_meta_train = y.iloc[train_idx]
    y_meta_val = y.iloc[val_idx]

    # Train meta-model (Ridge)
    meta_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    meta_model.fit(X_meta_train, y_meta_train)
    preds_meta = meta_model.predict(X_meta_val)
    meta_oof[val_idx] = preds_meta

    rmse_meta = np.sqrt(mean_squared_error(y_meta_val, preds_meta))
    meta_scores.append(rmse_meta)
    print(f"Meta-model fold {fold+1} RMSE: {rmse_meta:.5f}")

print(f"\nMeta-model OOF log RMSE: {np.mean(meta_scores):.5f} ± {np.std(meta_scores):.5f}")

Meta-model fold 1 RMSE: 0.09831
Meta-model fold 2 RMSE: 0.12841
Meta-model fold 3 RMSE: 0.10654
Meta-model fold 4 RMSE: 0.10329
Meta-model fold 5 RMSE: 0.13371
Meta-model fold 6 RMSE: 0.10218
Meta-model fold 7 RMSE: 0.12271
Meta-model fold 8 RMSE: 0.10309
Meta-model fold 9 RMSE: 0.10519
Meta-model fold 10 RMSE: 0.08905

Meta-model OOF log RMSE: 0.10925 ± 0.01350


Optionally, you can also compute the original‑scale RMSE:

In [6]:
meta_rmse_orig = rmse(inverse_log_transform(y), inverse_log_transform(meta_oof))
print(f"Meta-model OOF original RMSE: {meta_rmse_orig:.2f}")

Meta-model OOF original RMSE: 20209.83


## 6. Compare with Simple Blend and Best Base Model

From 04_modeling_baseline.ipynb, the simple blend (LGB + Ridge) gave log RMSE ~0.1128 and original RMSE ~21,180. Let's see how stacking compares.

In [7]:
best_base = min([(np.sqrt(mean_squared_error(y, oof_preds[name])), name) for name in base_models])
print(f"Best base model: {best_base[1]} with log RMSE = {best_base[0]:.5f}")
print(f"Simple blend (from notebook 04) log RMSE ≈ 0.1128")
print(f"Stacking log RMSE = {np.mean(meta_scores):.5f}")

Best base model: ridge with log RMSE = 0.11341
Simple blend (from notebook 04) log RMSE ≈ 0.1128
Stacking log RMSE = 0.10925


## 7. Train Final Meta-Model on Full OOF Predictions

To predict on the test set, we need to:

- For each fold, we have base models trained on that fold's training data.
- For the test set, average predictions from all folds (or train final models on full data, but that's a separate approach).  

### Standard stacking procedure for test set:

1. For each fold, use the base models (trained on that fold's training data) to predict on the test set.
2. This gives, for each fold, a set of test predictions from each base model.
3. Average these test predictions across folds to get final base model test predictions.
4. Apply the meta-model (trained on the full OOF predictions) to the averaged base predictions.

Alternatively, retraining each base model on the full training data is possible, but to stay consistent with OOF methodology and avoid overfitting, we use the fold-averaging approach.

In [11]:
# 7. Train Final Meta-Model on Full OOF Predictions

# Load test data
test_df = load_test()
test_ids = test_df["Id"]
X_test_raw = test_df.drop("Id", axis=1)

# Initialize arrays to store test predictions from each base model across folds
test_preds_base = {name: np.zeros(len(X_test_raw)) for name in base_models.keys()}

for fold_dict in fold_models:
    ft = fold_dict['ft']
    ohe = fold_dict['ohe']
    num_cols = fold_dict['num_cols']
    cat_cols = fold_dict['cat_cols']

    # Transform raw test data using the fold's FeatureTransformer
    X_test_t = ft.transform(X_test_raw)

    # One‑hot encode categorical columns using the fold's OHE
    X_test_cat = ohe.transform(X_test_t[cat_cols])
    X_test_num = X_test_t[num_cols].values
    X_test_proc = np.hstack([X_test_num, X_test_cat])

    # Predict with each base model
    for name in base_models.keys():
        model = fold_dict[name]
        preds = model.predict(X_test_proc)
        test_preds_base[name] += preds

# Average across folds
for name in test_preds_base:
    test_preds_base[name] /= len(fold_models)

# Train final meta-model on full OOF predictions
X_meta_full = np.column_stack([oof_preds[name] for name in base_models.keys()])
meta_final = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta_final.fit(X_meta_full, y)

# Prepare test meta features: stack the averaged base predictions
X_test_meta = np.column_stack([test_preds_base[name] for name in base_models.keys()])
test_preds_meta = meta_final.predict(X_test_meta)

# Inverse log transform to get original SalePrice
test_preds_original = inverse_log_transform(test_preds_meta)

## 8. Create Submission

In [12]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_preds_original
})
submission.to_csv(config.SUBMISSION_FILE, index=False)
print(f"Submission saved to {config.SUBMISSION_FILE}")

Submission saved to /home/madzimest/Documents/ReactJS/MyPortifolio/DataScience/DS/notebooks/submission.csv


In [13]:
import joblib
from pathlib import Path

# Define where to save the stacking pipeline
DEPLOY_DIR = config.PROJECT_ROOT / "deployment"
DEPLOY_DIR.mkdir(exist_ok=True)

# Save the fold models, meta-model, and other artifacts
stacking_pipeline = {
    'fold_models': fold_models,
    'meta_model': meta_final,
    'base_model_names': list(base_models.keys()),
    'feature_names': X_raw.columns.tolist()  # optional, for reference
}

joblib.dump(stacking_pipeline, DEPLOY_DIR / "stacking_pipeline.pkl")
print(f"Stacking pipeline saved to {DEPLOY_DIR / 'stacking_pipeline.pkl'}")

Stacking pipeline saved to /home/madzimest/Documents/ReactJS/MyPortifolio/DataScience/DS/deployment/stacking_pipeline.pkl
